# Pipeline EHBG-FACS · 05 · Propuesta (HBG-GFlowNet + GFACS + ENN)

**Paradigma 5 — muestreo híbrido GFlowNet de Balance Híbrido + colonia de hormigas.**

La propuesta de la tesis. Una **GFlowNet de Balance Híbrido (HBG)** —Attention Model que parametriza P_F, P_B y el flujo F_θ(s), entrenada con el objetivo híbrido **TB+DB** (ponderado por λ_DB) y **recompensa sensible al riesgo** R(x) ∝ exp(−CVaR_α/T)— acopla su matriz heurística a un **muestreador de Colonia de Hormigas (GFACS)** con búfer de repetición fuera de política. Busca simultáneamente **bajo costo y alta factibilidad** (la región que ningún paradigma base ocupa). La variante **ENN** activa la cabeza epinet para guiar la exploración con incertidumbre epistémica (Fase 5).

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50, 100, 200, 300]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = 5   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Entrenar EHBG-FACS (GPU)
Entrena la GFlowNet HBG con recompensa CVaR + refinamiento GFACS + replay off-policy, sobre un banco de entrenamiento fijo (semillas disjuntas del de evaluación). Hiperparámetros clave (Cuadro 2 del anteproyecto): `lam_db` (TB↔DB), `temperature` (suavizado de la política), `rho` (evaporación ACO).

In [ ]:
from svrplab.solvers.ehbg_facs import EHBGFACS
facs = EHBGFACS(train_sizes=(10,20), n_train=64, epochs=40, embed_dim=128,
                lam_db=0.5, temperature=2.0, batch=16, refine_every=5,
                infer_ants=16, infer_iters=12, infer_realizations=40,
                device=env.device, models_dir=env.paths.models, verbose=True)
df = runner.run_solver(facs, "ehbg-facs", bank, env, proto, verbose=True)
df

## (Opcional, Fase 5) Variante epistémica EHBG-FACS-ENN

In [ ]:
from svrplab.solvers.ehbg_facs import EHBGFACSEpistemic
facs_enn = EHBGFACSEpistemic(train_sizes=(10,20), n_train=64, epochs=40, embed_dim=128,
                             lam_db=0.5, temperature=2.0, batch=16, refine_every=5,
                             infer_ants=16, infer_iters=12, infer_realizations=40,
                             device=env.device, models_dir=env.paths.models, verbose=True)
df_enn = runner.run_solver(facs_enn, "ehbg-facs-enn", bank, env, proto, verbose=True)
df_enn

## Curvas de entrenamiento (TB / DB / CVaR) y figuras

In [ ]:
import matplotlib.pyplot as plt
h = getattr(facs, "history", {})
if h:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
    ax[0].plot(h["tb"]); ax[0].set_title("Balance de Trayectoria (TB)")
    ax[1].plot(h["db"], c="orange"); ax[1].set_title("Balance Detallado (DB)")
    ax[2].plot(h["cvar"], c="crimson"); ax[2].set_title("CVaR medio (recompensa)")
    for a in ax: a.set_xlabel("paso")
    plt.tight_layout(); plt.show()
display(metrics.aggregate_by_size(df))
inst = bank[SIZES[0]][0]
sol = facs.solve(inst, num_realizations=proto.realizations)
viz.plot_routes(inst, sol.routes, title=f"EHBG-FACS · n={SIZES[0]}"); plt.show()

**Interpretación.** EHBG-FACS combina el muestreo adaptativo (diversidad de la GFlowNet) con el refinamiento poblacional del ACO y una recompensa de cola (CVaR), apuntando a la **esquina ideal** (bajo costo *y* alta factibilidad) que el informe técnico identificó vacía. Compáralo con los baselines en el notebook 06.